<a href="https://colab.research.google.com/github/25ds3000042/LangChain-Chains-Retrieval-and-RAG/blob/main/day2_11_oct_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install -q langchain langchain-core langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install -q langchain-groq langchain-huggingface sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.0 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    api_key = userdata.get("GROQ_API_KEY")
)


In [ ]:
response = llm.invoke("What is langchain? Answer in one paragraph only")
response.content

'LangChain is an open‑source framework for building applications that integrate large language models (LLMs) with external data sources, APIs, and custom logic, enabling developers to create more dynamic, context‑aware, and interactive AI-driven experiences such as chatbots, agents, and retrieval‑augmented generation pipelines; it provides modular components for prompt management, memory, chaining, tooling, and evaluation, allowing seamless orchestration of LLM calls, data retrieval, and user-defined functions within a single, extensible codebase.'

In [ ]:
response = llm.invoke("What new product line was launched in 2024 in Acme Inc?")
print(response.content)

I’m sorry, but I can’t help with that.


In [ ]:
response = llm.invoke("How frequently does the HR team of Acme Inc meet and what do they assess?")
print(response.content)

I’m sorry, but I can’t provide that information.


In [ ]:
# RAG

# 1. Load the documents for retrieval.
# 2. Split the documents
# 3. Produce embeddings for each of the splits
# 4. Store the embeddings in a vectorDB
# 5. Create a retriever to retrieve the embeddings from the VectorDB
# 6. Combine the LLM and the retriever and produce results.

In [ ]:
# Document

from langchain_core.documents import Document

doc1 = Document(page_content = "This is the first document.", metadata = {"source": "first_file"})

In [ ]:
doc1

Document(metadata={'source': 'first_file'}, page_content='This is the first document.')

# using a txt file

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("RAG_file.txt")

docs1 = loader.load()

docs1

[Document(metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki, Finland. It specializes in anti-gravity footwear and rocket-powered pogo sticks.\n\nIn 2024, Acme released a new product line: "Jet Sneakers", designed for low-orbit recreational use.\n\nAs per internal policy, Acme\'s HR team meets every 2 weeks to assess wellness metrics of staff based on holographic surveys.\n')]

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("RAG_file.pdf")

docs2 = loader.load()

docs2

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20250730135006', 'source': 'RAG_file.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Acme Inc. was founded in 1987 in Helsinki, Finland. It specializes in anti-gravity footwear and\nrocket-powered pogo sticks. In 2024, Acme released a new product line: "Jet Sneakers",\ndesigned for low-orbit recreational use. As per internal policy, Acme\'s HR team meets every 2\nweeks to assess wellness metrics of staff based on holographic surveys.')]

In [ ]:
# CSVLoader
# UnstructuredHTMLLoader


In [ ]:
# all_docs = doc1 + docs2 + .....

# Split

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 50, chunk_overlap = 15)

In [ ]:
chunks = splitter.split_documents(docs1)

In [ ]:
chunks

[Document(metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='in Helsinki, Finland. It specializes in'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='specializes in anti-gravity footwear and'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='footwear and rocket-powered pogo sticks.'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='line: "Jet Sneakers", designed for low-orbit'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='for low-orbit recreational use.'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content="As per internal policy, Acme's HR team meets"),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='HR team meets every 2 weeks to assess wellness'),
 Document(metadata={'source': 'RAG_file.tx

In [ ]:
print(splitter._chunk_size)
print(splitter._chunk_overlap)

50
15


# Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectors = embedding_model.embed_documents([doc.page_content for doc in chunks])

In [ ]:
len(vectors)

11

In [ ]:
len(vectors[0])

384

# store embeddings in a vectorDB

In [ ]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embedding_model)

In [ ]:
vectorstore

In [ ]:
retrieved_docs = vectorstore.similarity_search("What was the launch date?", k = 2)
retrieved_docs

[Document(id='addd6aef-df86-4f20-9202-192e45648ade', metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
 Document(id='6d84bc96-4d27-4b73-9dfd-f580d3be7406', metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,')]

In [ ]:
retrieved_docs = vectorstore.similarity_search_with_score("What was the launch date?", k = 5)
retrieved_docs

[(Document(id='addd6aef-df86-4f20-9202-192e45648ade', metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
  np.float32(1.4024287)),
 (Document(id='6d84bc96-4d27-4b73-9dfd-f580d3be7406', metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,'),
  np.float32(1.5077162)),
 (Document(id='bf0c71af-23b7-45c5-b436-0a0c5065d9af', metadata={'source': 'RAG_file.txt'}, page_content='line: "Jet Sneakers", designed for low-orbit'),
  np.float32(1.5766808)),
 (Document(id='0db80738-69ed-4f6f-b361-e00a36c197df', metadata={'source': 'RAG_file.txt'}, page_content='for low-orbit recreational use.'),
  np.float32(1.6928246)),
 (Document(id='6407a7ed-1b57-4739-9d7a-0e01a88aa2d0', metadata={'source': 'RAG_file.txt'}, page_content='footwear and rocket-powered pogo sticks.'),
  np.float32(1.7089806))]

# create retriever

In [ ]:
retriever = vectorstore.as_retriever() # search_kwargs = {"k": 3}

In [ ]:
retriever.invoke("What was the launch date?")

[Document(id='addd6aef-df86-4f20-9202-192e45648ade', metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
 Document(id='6d84bc96-4d27-4b73-9dfd-f580d3be7406', metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,'),
 Document(id='bf0c71af-23b7-45c5-b436-0a0c5065d9af', metadata={'source': 'RAG_file.txt'}, page_content='line: "Jet Sneakers", designed for low-orbit'),
 Document(id='0db80738-69ed-4f6f-b361-e00a36c197df', metadata={'source': 'RAG_file.txt'}, page_content='for low-orbit recreational use.')]

# Create RAG chain

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
template = """
Use the following context to answer the question.
Use three sentences maximum and keep the answer concise.

Context: {context}

Question: {question}

Answer:
"""

prompt = PromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
query = "What new product line was launched in 2024 in Acme Inc?"

response = rag_chain.invoke(query)
response

'Acme Inc. launched the “Jet Sneakers” product line in 2024.'

In [ ]:
query = "When and where was Acme Inc. founded?"

response = rag_chain.invoke(query)
response

'Acme Inc. was founded in 1987 in Helsinki.'